# Sequence Retrieval with the PBI Package

This notebook demonstrates all the ways to retrieve sequences and data from the PBI package, from simple metadata queries to full sequence retrieval and batch iteration.

## What You'll Learn
1. **Database Connection** - How `quick_connect()` works internally
2. **Database Statistics** - Overview of available data
3. **Querying Phage Metadata** - SQL-style filtering
4. **Querying Host Metadata** - Filter by assembly quality, species
5. **Querying Phage-Host Pairs** - Metadata-only joins
6. **Pagination** - LIMIT and OFFSET for large datasets
7. **Retrieving Phage Sequences** - DNA sequences from FASTA
8. **Retrieving Host Sequences** - Bacterial genome sequences
9. **Full Genome Retrieval** - Multi-contig assembly with `get_host_genome()` and `get_phage_genome()`
10. **Retrieving Protein Sequences** - Phage protein sequences
11. **Getting DataFrames** - Tabular results with sequences
12. **Batch Iteration** - Memory-efficient processing
13. **Understanding Warnings** - Missing sequence messages explained

## How PBI Works Internally

```
pbi.quick_connect()
  |
  +-- Finds database file (auto-detected from common paths)
  +-- Opens DuckDB connection (fast, in-process SQL)
  +-- Locates FASTA files (phage, protein, host sequences)
  +-- Builds pyfaidx indexes if not present (.fai files)
  |
  v
SequenceRetriever
  |
  +-- get_phage_metadata()     -> SQL query -> DataFrame
  +-- get_host_metadata()      -> SQL query -> DataFrame
  +-- get_phage_host_pairs()   -> SQL + FASTA lookup -> DataFrame with sequences
  +-- get_host_genome()        -> Full genome (multi-contig safe)
  +-- get_host_genome_stats()  -> Contig-level stats
  +-- get_phage_genome()       -> Full phage genome
  +-- create_streaming_dataset() -> Iterator over samples
  +-- create_indexed_dataset()   -> Random-access dataset
```

## Setup & Imports

In [1]:
import sys
from pathlib import Path
import time

project_root = Path.cwd().parent
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import pbi
import pandas as pd

# Set pandas display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("PBI package imported successfully")
print(f"Version: {pbi.__version__ if hasattr(pbi, '__version__') else 'dev'}")

import os

results_root = Path(os.getenv('PBI_RESULTS_DIR', '/results'))
try:
    results_root.mkdir(parents=True, exist_ok=True)
except Exception:
    results_root = project_root / 'analysis_results'
    results_root.mkdir(parents=True, exist_ok=True)

NOTEBOOK_RESULTS_DIR = results_root / '02_sequence_retrieval'
TABLES_DIR = NOTEBOOK_RESULTS_DIR / 'tables'
FIGURES_DIR = NOTEBOOK_RESULTS_DIR / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Notebook results directory: {NOTEBOOK_RESULTS_DIR}")


PBI package imported successfully
Version: 0.2.0
Notebook results directory: /results/02_sequence_retrieval


## Database Connection

`pbi.quick_connect()` searches for the PBI database and FASTA files automatically. It:
1. **Looks in common locations** for the `.duckdb` database file
2. **Opens a DuckDB connection** — DuckDB is an embedded database (like SQLite but faster for analytics)
3. **Locates FASTA files** for phage sequences, protein sequences, and host sequences
4. **Builds FASTA indexes** (`.fai` files) using pyfaidx if they don't exist yet

The connection is nearly instant because FASTA files are indexed — sequences are retrieved with direct byte-offset reads, not by scanning the entire file.

In [2]:
# Quick connect - auto-detects database and FASTA files
print("Connecting to PBI database...")
start = time.time()
retriever = pbi.quick_connect()
elapsed = time.time() - start

print(f"Connected in {elapsed:.2f} seconds")
print(f"Database connection: {retriever.conn}")
print(f"Phage FASTA ready: {retriever._phage_fasta is not None}")
print(f"Host data available: {retriever._has_host_data if hasattr(retriever, '_has_host_data') else 'unknown'}")

Connecting to PBI database...


2026-04-20 10:38:20,784 - INFO - 📂 Checking FASTA index files:
2026-04-20 10:38:20,786 - INFO -    Phage index: True (52570.4 KB)
2026-04-20 10:38:20,787 - INFO -    Protein index: True (1432185.2 KB)
2026-04-20 10:38:20,790 - INFO - 📂 Loaded private phage mapping for 2 sources: ['test_private', 'test_private_2']
2026-04-20 10:38:20,791 - INFO - 📂 Using host mapping file: /data/processed/sequences/host_fasta_mapping.json
2026-04-20 10:38:20,799 - INFO -    Loaded mapping for 5542 hosts
2026-04-20 10:38:20,800 - INFO - 📂 Connecting to database: /data/processed/databases/phage_database_optimized.duckdb
2026-04-20 10:38:20,857 - INFO - 🔄 Starting background FASTA loading...
2026-04-20 10:38:20,859 - INFO - 🔄 [Background] Loading phage FASTA: /data/processed/sequences/all_phages.fasta
2026-04-20 10:38:20,859 - INFO - ✅ Initialization complete (FASTA loading in background)


Connected in 0.39 seconds
Database connection: <_duckdb.DuckDBPyConnection object at 0x7f9509f7f9b0>
Phage FASTA ready: False
Host data available: True


## Database Statistics

The `get_stats()` method runs a few COUNT queries and checks FASTA file sizes. It returns a nested dictionary with:
- `stats['database']`: Record counts from DuckDB tables
- `stats['fasta']`: Sequence counts from indexed FASTA files

The difference between database and FASTA counts reflects the coverage — not all database records have associated sequences.

In [3]:
# Get comprehensive statistics
stats = retriever.get_stats()

print("Database Statistics:")
print("=" * 60)
print(f"Phages:   {stats['database']['phages']:,}")
print(f"Proteins: {stats['database']['proteins']:,}")
print(f"Hosts:    {stats['database']['hosts']:,}")
print(f"Phage-Host Associations: {stats['database']['phage_host_associations']:,}")
print()
print("FASTA Files:")
print(f"  Phage sequences:   {stats['fasta']['phages']:,}")
print(f"  Protein sequences: {stats['fasta']['proteins']:,}")
if 'hosts' in stats['fasta']:
    print(f"  Host sequences:    {stats['fasta']['hosts']:,}")

2026-04-20 10:38:22,016 - INFO - ⏳ Waiting for FASTA loading to complete...
2026-04-20 10:38:29,082 - INFO -    ✅ Phage FASTA loaded in 8.22s (873,717 sequences)
2026-04-20 10:38:29,083 - INFO - 🔄 [Background] Loading protein FASTA: /data/processed/sequences/all_proteins.fasta


KeyboardInterrupt: 

## Querying Phage Metadata

`retriever.get_phage_metadata()` executes a SELECT query against the `fact_phages` table. You can pass any SQL WHERE clause (without the `WHERE` keyword), or use `LIMIT`/`OFFSET` directly.

**Available columns typically include:**
- `Phage_ID`: Unique identifier
- `Length`: Genome length in base pairs
- `GC_content`: GC percentage
- `Taxonomy`: Taxonomic classification
- `Completeness`: Genome completeness status
- `Lifestyle`: Lytic, lysogenic, or unknown
- `Source_DB`: Which database the record came from
- `Host`: Recorded host organism

In [ ]:
# Get all phage metadata (no filter)
phage_metadata = retriever.get_phage_metadata()
print(f"Total phages: {len(phage_metadata):,}")
print(f"Columns: {list(phage_metadata.columns)}")
phage_metadata.head()

In [ ]:
# Filter: only complete phages
try:
    complete_phages = retriever.get_phage_metadata(
        where_clause="Completeness LIKE 'Complete'" # Does not work
    )
    print(f"Complete phages: {len(complete_phages):,}")
    print(set(complete_phages['Completeness']))
except Exception:
    # Try with different capitalization
    completeness_values = phage_metadata['Completeness'].value_counts()
    print("Available completeness values:")
    print(completeness_values.head(10))

In [ ]:
# Filter: large phages (>100 kb)
large_phages = retriever.get_phage_metadata(
    where_clause="Length > 100000"
)
print(f"Large phages (>100 kb): {len(large_phages):,}")
if len(large_phages) > 0:
    print(f"Largest: {large_phages['Length'].max():,} bp")
    print(large_phages[['Phage_ID', 'Length', 'GC_content', 'Taxonomy']].head())

## Querying Host Metadata

`retriever.get_host_metadata()` queries the `dim_hosts` table. Host records represent bacterial genomes with assembly information from NCBI.

**Available columns typically include:**
- `Host_ID`: Unique identifier (usually NCBI assembly accession)
- `Species_Name`: Bacterial species
- `Assembly_Level`: Complete Genome, Scaffold, Contig, etc.
- `Genome_Length`: Total genome size in bp
- `GC_Content`: Host GC percentage

In [ ]:
# Get host metadata with filter
try:
    host_metadata = retriever.get_host_metadata(
        where_clause="Assembly_Level = 'Complete Genome'",
        limit=10
    )
    print(f"Complete genome hosts (sample): {len(host_metadata)}")
    print(host_metadata[['Species_Name', 'Assembly_Level', 'Genome_Length', 'GC_Content']].head())
except Exception as e:
    print(f"Host metadata query: {e}")
    # Try without filter
    host_metadata = retriever.get_host_metadata()
    print(f"All hosts: {len(host_metadata):,}")
    print(f"Assembly levels: {host_metadata['Assembly_Level'].value_counts().head()}")

## Querying Phage-Host Pairs (Metadata Only)

`retriever.get_phage_host_metadata()` (or `get_phage_host_pairs(sequences=False)`) performs a SQL JOIN between phage and host tables through the association table. This returns metadata without loading sequences — fast for large queries.

**Use this when you need:**
- Statistics about pairs
- Filtering before sequence retrieval
- Building feature tables from metadata alone

In [ ]:
# Query phage-host pairs (metadata only, no sequences)
try:
    pairs_meta = retriever.get_phage_host_metadata(
        where_clause="Length > 10000 AND Species_Name LIKE 'Escherichia%'",
        limit=20
    )
    print(f"Escherichia-infecting phages (>10kb): {len(pairs_meta)}")
    if len(pairs_meta) > 0:
        print(pairs_meta[['Phage_ID', 'Phage_Length', 'Host_Species', 'Host_Assembly_Level']].head())
except Exception as e:
    print(f"Note: {e}")
    pairs_meta = retriever.get_phage_host_metadata(limit=5)
    print(f"Sample pairs (no filter): {len(pairs_meta)}")
    if len(pairs_meta) > 0:
        print(pairs_meta.head())

## LIMIT and OFFSET for Pagination

For large datasets, use LIMIT and OFFSET to process data in pages. This is more memory-efficient than loading everything at once.

The `where_clause` parameter accepts these directly at the end of the clause:

In [ ]:
# Pagination with LIMIT and OFFSET
print("Pagination example:")
print("-" * 60)

# First 5 phages
batch1 = retriever.get_phage_metadata(where_clause="LIMIT 5")
print(f"Batch 1 (first 5): {list(batch1['Phage_ID'])}")

# Next 5 phages
batch2 = retriever.get_phage_metadata(where_clause="LIMIT 5 OFFSET 5")
print(f"Batch 2 (next 5): {list(batch2['Phage_ID'])}")

# Verify they don't overlap
overlap = set(batch1['Phage_ID']) & set(batch2['Phage_ID'])
print(f"Overlap between batches: {len(overlap)} (expected 0)")

## Retrieving Phage Sequences

`retriever.get_phage_host_pairs()` is the main method for retrieving sequences. It:
1. Runs a SQL JOIN to get metadata
2. For each row, looks up the phage sequence in the FASTA file using the `.fai` index
3. Looks up the host sequence in the host FASTA files
4. Returns a DataFrame with both metadata and sequences

**Under the hood (pyfaidx):** The `.fai` index stores byte offsets for each sequence. Retrieval is O(1) — it seeks directly to the right position without reading the entire file.

**Note:** Pairs where phage or host sequences are not found are automatically skipped with a warning. This is expected behavior.

In [ ]:
# Get phage-host pairs with sequences
print("Retrieving phage-host pairs with sequences...")
start = time.time()

pairs = retriever.get_phage_host_pairs(
    where_clause="LIMIT 5"
)

elapsed = time.time() - start
print(f"Retrieved {len(pairs)} pairs in {elapsed:.2f}s")
print(f"\nColumns: {list(pairs.columns)}")

if len(pairs) > 0:
    print(f"\nSample data:")
    print(pairs[['Phage_ID', 'Species_Name', 'Phage_Length', 'Host_Length']].head())

    print(f"\nSequence preview (first pair):")
    row = pairs.iloc[0]
    phage_seq = row.get('Phage_Sequence', '')
    host_seq = row.get('Host_Sequence', '')
    if phage_seq:
        print(f"  Phage {row['Phage_ID'][:30]}: {phage_seq[:60]}...")
    if host_seq:
        print(f"  Host {str(row.get('Species_Name', ''))[:30]}: {host_seq[:60]}...")

In [ ]:
# Filter by taxonomy for specific phage families
print("Retrieving Siphoviridae phage-host pairs...")
sipho_pairs = retriever.get_phage_host_pairs(
    where_clause="Taxonomy LIKE '%Siphoviridae%' LIMIT 10"
)
print(f"Found {len(sipho_pairs)} Siphoviridae phage-host pairs")
if len(sipho_pairs) > 0:
    print("\nTaxonomy distribution:")
    print(sipho_pairs['Phage_Taxonomy'].value_counts() if 'Phage_Taxonomy' in sipho_pairs.columns else sipho_pairs['Taxonomy'].value_counts())

## Retrieving Host Sequences

Host sequences are stored either in a combined FASTA file or as individual per-genome files (depending on the pipeline configuration). The `SequenceRetriever` handles both modes transparently.

When you call `get_phage_host_pairs()`, host sequences are retrieved automatically. You can also access the host FASTA directly via the retriever's internal attributes.

In [ ]:
# Inspect host sequence availability
print("Host sequence information:")
print("-" * 60)

if len(pairs) > 0:
    has_host_seq = pairs['Host_Sequence'].notna() & (pairs['Host_Sequence'] != '')
    print(f"Pairs with host sequences: {has_host_seq.sum()}/{len(pairs)}")
    if has_host_seq.any():
        sample = pairs[has_host_seq].iloc[0]
        host_seq = sample['Host_Sequence']
        print(f"\nSample host sequence:")
        print(f"  Species: {sample.get('Species_Name', 'N/A')}")
        print(f"  Length: {len(host_seq):,} bp")
        print(f"  First 80 bp: {host_seq[:80]}")
else:
    print("No pairs retrieved - check database content")

## Full Genome Retrieval (Multi-Contig Support)

PR #64 added three dedicated methods for retrieving **complete** host or phage genomes,
even when the assembly is split across multiple scaffolds, chromosomes, or contigs:

| Method | Description |
|---|---|
| `get_host_genome(host_id, mode, gap, order)` | Retrieve and assemble a host genome |
| `get_host_genome_stats(host_id, order)` | Contig statistics without loading sequences |
| `get_phage_genome(phage_id, mode, gap, order)` | Retrieve a phage genome (usually single-contig) |

### Assembly modes

All three methods share the same `mode` parameter:

| `mode` | Returns | Use case |
|---|---|---|
| `"concat"` *(default)* | Single `str` — all contigs joined | Training sequence models |
| `"first"` | Single `str` — largest contig only | Quick inspection; legacy behaviour |
| `"list"` | `list[str]` — one entry per contig | Per-contig analysis |
| `"dict"` | `dict[str, str]` — header → sequence | Selective access by contig name |

You can also pass `gap=N` (default `0`) to insert `N` `"N"` characters between contigs
when using `mode="concat"`. This is useful when downstream tools expect gaps between
assembly scaffolds.

The same functionality is also available as a standalone utility:

```python
from pbi.fasta_utils import assemble_genome, get_genome_stats
```

And `get_phage_host_pairs()` / `get_phage_host_pairs_iterator()` both accept
`host_contig_mode=` to control how host sequences are assembled for **bulk** retrieval.


In [ ]:
# ── Test: get_host_genome_stats() ─────────────────────────────────────────
# Inspect contig fragmentation of a host genome WITHOUT loading the full
# sequence into memory.  Useful for choosing an appropriate assembly mode.
print("Host genome contig statistics")
print("=" * 60)

if hasattr(retriever, '_has_host_data') and retriever._has_host_data:
    # Pick the first host ID that has sequence data available
    host_ids = []
    try:
        host_meta = retriever.get_host_metadata()
        host_ids = host_meta['Host_ID'].dropna().tolist() if 'Host_ID' in host_meta.columns else []
    except Exception:
        pass

    found_host = None
    for hid in host_ids[:20]:  # probe at most 20 candidates
        try:
            stats = retriever.get_host_genome_stats(str(hid))
            found_host = str(hid)
            break
        except (KeyError, ValueError):
            continue

    if found_host:
        stats = retriever.get_host_genome_stats(found_host)
        print(f"Host ID          : {found_host}")
        print(f"Contig count     : {stats['contig_count']}")
        print(f"Total length     : {stats['total_length']:,} bp")
        print(f"Contig lengths   : {stats['lengths']}")
        if stats['contig_count'] > 1:
            print(f"  → Multi-contig assembly: use mode='concat' to get the full genome")
        else:
            print(f"  → Single-contig assembly: all modes return the same result")
    else:
        print("No host sequence data available in this environment.")
        print("(Run with a real PBI database and host FASTA files to see results.)")
else:
    print("Host data not configured for this retriever instance.")
    print("Tip: use pbi.quick_connect() with a directory that contains host FASTA files.")


In [ ]:
# ── Test: get_host_genome() — all assembly modes ───────────────────────────
print("get_host_genome() — assembly mode comparison")
print("=" * 60)

if hasattr(retriever, '_has_host_data') and retriever._has_host_data and found_host:
    # mode='concat' (default) — single string, all contigs joined
    genome_concat = retriever.get_host_genome(found_host)          # mode='concat', gap=0
    print(f"mode='concat' (default)")
    print(f"  type   : {type(genome_concat).__name__}")
    print(f"  length : {len(genome_concat):,} bp")
    print(f"  preview: {genome_concat[:60]}...")
    print()

    # mode='concat' with 100-N gap between scaffolds
    genome_gapped = retriever.get_host_genome(found_host, mode="concat", gap=100)
    print(f"mode='concat', gap=100")
    print(f"  type   : {type(genome_gapped).__name__}")
    print(f"  length : {len(genome_gapped):,} bp")
    stats = retriever.get_host_genome_stats(found_host)
    expected_extra = (stats['contig_count'] - 1) * 100
    assert len(genome_gapped) == len(genome_concat) + expected_extra, \
        f"Gap length mismatch: {len(genome_gapped)} != {len(genome_concat)} + {expected_extra}"
    print(f"  ✓ gap length verified (+{expected_extra} N characters)")
    print()

    # mode='first' — only the largest contig
    genome_first = retriever.get_host_genome(found_host, mode="first")
    print(f"mode='first' (largest contig)")
    print(f"  type   : {type(genome_first).__name__}")
    print(f"  length : {len(genome_first):,} bp")
    assert isinstance(genome_first, str), "mode='first' should return a str"
    print(f"  ✓ returns a plain string")
    print()

    # mode='list' — one string per contig
    genome_list = retriever.get_host_genome(found_host, mode="list")
    print(f"mode='list'")
    print(f"  type   : {type(genome_list).__name__}")
    print(f"  contigs: {len(genome_list)}")
    assert isinstance(genome_list, list), "mode='list' should return a list"
    assert len(genome_list) == stats['contig_count'], \
        f"Contig count mismatch: {len(genome_list)} != {stats['contig_count']}"
    total_from_list = sum(len(s) for s in genome_list)
    assert total_from_list == stats['total_length'], \
        f"Total length mismatch: {total_from_list} != {stats['total_length']}"
    print(f"  ✓ list length and total sequence length match stats")
    print()

    # mode='dict' — {header: sequence}
    genome_dict = retriever.get_host_genome(found_host, mode="dict")
    print(f"mode='dict'")
    print(f"  type   : {type(genome_dict).__name__}")
    print(f"  keys   : {list(genome_dict.keys())[:3]}{'...' if len(genome_dict) > 3 else ''}")
    assert isinstance(genome_dict, dict), "mode='dict' should return a dict"
    assert len(genome_dict) == stats['contig_count'], \
        f"Dict length mismatch: {len(genome_dict)} != {stats['contig_count']}"
    print(f"  ✓ dict key count matches contig count")
    print()

    print("All get_host_genome() mode assertions passed ✓")
else:
    print("Skipped: host data not available in this environment.")
    print("Example (pseudocode):")
    print("  genome = retriever.get_host_genome('GCF_000005845')          # str")
    print("  genome = retriever.get_host_genome('GCF_000005845', gap=100) # with N-gaps")
    print("  contigs = retriever.get_host_genome('GCF_000005845', mode='list')  # list")
    print("  seqmap  = retriever.get_host_genome('GCF_000005845', mode='dict')  # dict")


In [ ]:
# ── Test: get_phage_genome() ───────────────────────────────────────────────
# Phage genomes are almost always single-contig, but get_phage_genome() uses
# the same mode/gap/order API as get_host_genome() for consistency.
print("get_phage_genome() example")
print("=" * 60)

phage_ids = phage_metadata['Phage_ID'].dropna().tolist() if len(phage_metadata) > 0 else []

found_phage = None
for pid in phage_ids[:20]:
    try:
        seq = retriever.get_phage_genome(str(pid))
        found_phage = str(pid)
        break
    except (KeyError, AttributeError):
        continue

if found_phage:
    # Default: mode='concat'
    seq_concat = retriever.get_phage_genome(found_phage)
    seq_first  = retriever.get_phage_genome(found_phage, mode="first")
    seq_list   = retriever.get_phage_genome(found_phage, mode="list")
    seq_dict   = retriever.get_phage_genome(found_phage, mode="dict")

    print(f"Phage ID : {found_phage}")
    print(f"Length   : {len(seq_concat):,} bp")
    print(f"Preview  : {seq_concat[:60]}...")
    print()

    # Consistency checks for a typical single-contig phage
    assert isinstance(seq_concat, str),       "concat should return str"
    assert isinstance(seq_first,  str),       "first should return str"
    assert isinstance(seq_list,   list),      "list should return list"
    assert isinstance(seq_dict,   dict),      "dict should return dict"

    # For a single-contig phage all modes must yield the same sequence
    assert seq_concat == seq_first,           "concat == first for single-contig"
    assert seq_list == [seq_concat],          "list wraps the single sequence"
    assert seq_dict == {found_phage: seq_concat}, "dict maps id -> sequence"

    print("All get_phage_genome() assertions passed ✓")
else:
    print("Skipped: no phage sequences available in this environment.")
    print("Example (pseudocode):")
    print("  seq = retriever.get_phage_genome('NC_000866')")
    print("  # For a typical single-contig phage this is equivalent to:")
    print("  seq = str(retriever.phage_fasta['NC_000866'][:].seq)")


In [ ]:
# ── Test: get_phage_host_pairs(host_contig_mode='concat') ─────────────────
# The host_contig_mode parameter controls how host sequences are assembled
# when fetching phage-host pairs.  Using 'concat' ensures that fragmented
# host assemblies are returned as a single full-genome string rather than
# just the first (largest) scaffold.
print("get_phage_host_pairs() — host_contig_mode comparison")
print("=" * 60)

try:
    pairs_first  = retriever.get_phage_host_pairs(
        where_clause="LIMIT 5",
        host_contig_mode="first",    # default: largest contig only
    )
    pairs_concat = retriever.get_phage_host_pairs(
        where_clause="LIMIT 5",
        host_contig_mode="concat",   # full genome: all contigs joined
    )

    print(f"Pairs retrieved: {len(pairs_first)}")

    if len(pairs_first) > 0 and len(pairs_concat) > 0:
        host_first_lens  = pairs_first['Host_Sequence'].dropna().str.len()
        host_concat_lens = pairs_concat['Host_Sequence'].dropna().str.len()

        print(f"\nhost_contig_mode='first'  — host sequence lengths: {host_first_lens.tolist()}")
        print(f"host_contig_mode='concat' — host sequence lengths: {host_concat_lens.tolist()}")

        # concat lengths must be >= first lengths (concat includes all contigs)
        for i, (fl, cl) in enumerate(zip(host_first_lens, host_concat_lens)):
            assert cl >= fl, \
                f"Row {i}: concat length ({cl}) < first length ({fl})"

        print()
        print("✓ host_contig_mode='concat' lengths are >= mode='first' lengths")
        print("  (equal when host is single-contig, larger when fragmented)")
    else:
        print("No pairs with host sequences available — skipping length comparison.")
except Exception as e:
    print(f"Note: {e}")
    print("Example (pseudocode):")
    print("  # Full host genomes in bulk retrieval:")
    print("  pairs = retriever.get_phage_host_pairs(host_contig_mode='concat')")
    print("  # Batch iterator with full host genomes:")
    print("  for batch in retriever.get_phage_host_pairs_iterator(host_contig_mode='concat'):")
    print("      process(batch)")


In [ ]:
# ── Example: fasta_utils.assemble_genome() and get_genome_stats() ─────────
# These are the low-level building blocks used by get_host_genome().
# You can call them directly when you already have an open pyfaidx.Fasta object.
from pbi.fasta_utils import assemble_genome, get_genome_stats
from pyfaidx import Fasta
import tempfile, os

print("fasta_utils — assemble_genome() and get_genome_stats()")
print("=" * 60)

# Build a tiny in-memory multi-contig FASTA for the demo
fasta_content = (
    ">contig_A\nACGTACGTACGTACGT\n"
    ">contig_B\nTTTTAAAACCCCGGGG\n"
    ">contig_C\nNNNNACGTACGT\n"
)
tmp_fasta = tempfile.NamedTemporaryFile(suffix=".fasta", delete=False, mode="w")
tmp_fasta.write(fasta_content)
tmp_fasta.close()

try:
    fasta_obj = Fasta(tmp_fasta.name)

    # --- get_genome_stats ---
    stats = get_genome_stats(fasta_obj)
    print(f"get_genome_stats():")
    print(f"  contig_count : {stats['contig_count']}")
    print(f"  lengths      : {stats['lengths']}")
    print(f"  total_length : {stats['total_length']}")
    assert stats['contig_count'] == 3
    assert stats['total_length'] == 16 + 16 + 12
    print(f"  ✓ contig_count=3, total_length=44")
    print()

    # --- assemble_genome mode='concat' (default) ---
    seq_concat = assemble_genome(fasta_obj)
    print(f"assemble_genome(mode='concat', gap=0):")
    print(f"  result : {repr(seq_concat)}")
    assert len(seq_concat) == 44
    print(f"  ✓ length={len(seq_concat)} (sum of all contigs)")
    print()

    # --- mode='concat' with gap ---
    seq_gapped = assemble_genome(fasta_obj, mode="concat", gap=5)
    print(f"assemble_genome(mode='concat', gap=5):")
    print(f"  result : {repr(seq_gapped)}")
    # 3 contigs → 2 gaps of 5 N's each
    assert len(seq_gapped) == 44 + 2 * 5
    assert "NNNNN" in seq_gapped
    print(f"  ✓ length={len(seq_gapped)} (44 + 2×5 gap N's)")
    print()

    # --- mode='first' → longest contig ---
    seq_first = assemble_genome(fasta_obj, mode="first")
    print(f"assemble_genome(mode='first'):")
    print(f"  result : {repr(seq_first)}")
    # contig_A and contig_B are both 16 bp; tie-broken alphabetically
    assert isinstance(seq_first, str) and len(seq_first) == 16
    print(f"  ✓ length=16 (largest contig)")
    print()

    # --- mode='list' ---
    seq_list = assemble_genome(fasta_obj, mode="list")
    print(f"assemble_genome(mode='list'):")
    print(f"  result : {seq_list}")
    assert isinstance(seq_list, list) and len(seq_list) == 3
    print(f"  ✓ list of 3 strings")
    print()

    # --- mode='dict' ---
    seq_dict = assemble_genome(fasta_obj, mode="dict")
    print(f"assemble_genome(mode='dict'):")
    print(f"  keys   : {list(seq_dict.keys())}")
    assert isinstance(seq_dict, dict) and len(seq_dict) == 3
    print(f"  ✓ dict of 3 entries")
    print()

    # --- order='file' ---
    seq_file_order = assemble_genome(fasta_obj, mode="list", order="file")
    print(f"assemble_genome(mode='list', order='file'):")
    print(f"  result : {seq_file_order}")
    assert isinstance(seq_file_order, list)
    print(f"  ✓ preserves FASTA file order")
    print()

    print("All fasta_utils assertions passed ✓")

finally:
    fasta_obj.close()
    os.unlink(tmp_fasta.name)
    # Remove index file created by pyfaidx
    fai = tmp_fasta.name + ".fai"
    if os.path.exists(fai):
        os.unlink(fai)


## Retrieving Protein Sequences

PBI stores phage protein sequences in a separate FASTA file. Proteins are annotated phage genes — multiple proteins per phage.

You can access protein sequences directly or filter proteins by phage ID using SQL.

In [ ]:
# Query protein metadata
try:
    proteins = retriever.get_protein_metadata(
        where_clause="LIMIT 5"
    )
    print(f"Sample protein records: {len(proteins)}")
    if len(proteins) > 0:
        print(f"Columns: {list(proteins.columns)}")
        print(proteins.head())
except AttributeError:
    # Fallback: query directly
    try:
        proteins_df = retriever.conn.execute(
            "SELECT * FROM dim_proteins LIMIT 5"
        ).fetchdf()
        print(f"Sample proteins from dim_proteins: {len(proteins_df)}")
        print(proteins_df.head())
    except Exception as e2:
        print(f"Protein table query: {e2}")
except Exception as e:
    print(f"Protein query: {e}")

## Getting Results as DataFrames

All retrieval methods return pandas DataFrames. This makes it easy to:
- Export to CSV/Parquet
- Merge with other datasets
- Apply pandas operations (groupby, pivot, merge)
- Feed into sklearn or other ML libraries

In [ ]:
# Example: analyze sequence lengths from retrieved pairs
if len(pairs) > 0:
    print("Sequence length analysis:")
    print("-" * 60)

    if 'Phage_Length' in pairs.columns:
        print(f"Phage lengths: min={pairs['Phage_Length'].min():,}, max={pairs['Phage_Length'].max():,}, mean={pairs['Phage_Length'].mean():,.0f}")

    if 'Host_Length' in pairs.columns:
        host_len = pairs['Host_Length'].dropna()
        if len(host_len) > 0:
            print(f"Host lengths: min={host_len.min():,}, max={host_len.max():,}, mean={host_len.mean():,.0f}")

    # Export example
    # pairs.to_csv('phage_host_pairs.csv', index=False)
    # pairs.to_parquet('phage_host_pairs.parquet', index=False)
    print("\nTo export: pairs.to_csv('output.csv', index=False)")

## Batch Iteration with `get_phage_host_pairs_iterator()`

For datasets too large to load all at once, use the batch iterator. It:
- Runs paginated SQL queries internally
- Yields one DataFrame batch at a time
- Fetches sequences for each batch before moving to the next
- Keeps memory constant regardless of total dataset size

**When to use:**
- Processing >10,000 pairs
- Computing statistics over the full database
- Writing results to disk incrementally

In [ ]:
# Batch iteration example
print("Batch iteration example:")
print("-" * 60)

batch_iterator = retriever.get_phage_host_pairs_iterator(
    where_clause="p.Length > 10000",
    batch_size=100
)

max_batches = 3
total_processed = 0
all_phage_lengths = []

for i, batch_df in enumerate(batch_iterator):
    if i >= max_batches:
        break

    batch_size = len(batch_df)
    total_processed += batch_size
    if 'Phage_Length' in batch_df.columns:
        all_phage_lengths.extend(batch_df['Phage_Length'].dropna().tolist())

    print(f"Batch {i+1}: {batch_size} pairs")
    if i == 0:
        print(f"  Columns: {list(batch_df.columns[:6])}...")

print(f"\nTotal processed: {total_processed} pairs")
if all_phage_lengths:
    import numpy as np
    print(f"Mean phage length: {np.mean(all_phage_lengths):,.0f} bp")

## Understanding Missing Sequences (The Warnings Explained)

When retrieving sequences, you may see log messages like:

```
WARNING - Host genome not found in mapping: GCF_000123456.1
WARNING - Phage sequence not in FASTA: PHAGE_XYZ_001  
WARNING - Skipping pair: host genome not available
```

**These are informational, not errors.** The PBI pipeline:
1. Downloads phage sequences from source databases
2. Downloads host genomes from NCBI based on phage metadata
3. Stores them in indexed FASTA files

But not every phage has a sequenced host, and not every host genome was successfully downloaded. The retriever skips unavailable sequences and continues.

**To track which pairs are skipped**, use the `missing_hosts_csv` parameter:

In [ ]:
# Track missing sequences during retrieval
import tempfile
import os

missing_csv = Path(tempfile.mktemp(suffix='_missing.csv'))

try:
    dataset = retriever.create_indexed_dataset(
        where_clause="LIMIT 50",
        missing_hosts_csv=str(missing_csv)
    )
    print(f"Created dataset with {len(dataset)} samples")

    # Access a few samples (triggers sequence lookup)
    for i in range(min(5, len(dataset))):
        _ = dataset[i]

    if missing_csv.exists():
        missing_df = pd.read_csv(missing_csv)
        print(f"Missing sequences logged: {len(missing_df)}")
        if len(missing_df) > 0:
            print("\nFailure reasons:")
            print(missing_df['Failure_Reason'].value_counts())
    else:
        print("All sequences found - no missing data")
except Exception as e:
    print(f"Note: {e}")
finally:
    if missing_csv.exists():
        os.unlink(str(missing_csv))

## Private Data: Sequence Retrieval for `test_private`

This section verifies that the synthetic phage and host sequences shipped in
`private_data/test_private/` are correctly ingested and retrievable via the
same API as public PhageScope data.

`host.fasta` is required for private datasets: every `Host_ID` in
`metadata.csv` must resolve to a private host FASTA sequence.

Expected results:
- 5 phage-host pairs from `test_private`
- 5 phage sequences (`TEST_PHAGE_ALPHA` … `TEST_PHAGE_EPSILON`)
- 3 unique host sequences (`TEST_HOST_ECOLI_A`, `TEST_HOST_BSUBT_B`, `TEST_HOST_PAERUG_C`)
- `Phage_Source_Type` set to `private` for all retrieved rows


In [ ]:
# ── Verify test_private sequences are ingested ──────────────────────────────
print('=' * 60)
print('Private data sequence retrieval test (test_private)')
print('=' * 60)

# 1. Check metadata presence
private_phages = retriever.get_phage_metadata(where_clause="Source_DB = 'test_private'")
expected_phage_ids = {
    'TEST_PHAGE_ALPHA', 'TEST_PHAGE_BETA', 'TEST_PHAGE_GAMMA',
    'TEST_PHAGE_DELTA', 'TEST_PHAGE_EPSILON'
}
found_phage_ids = set(private_phages['Phage_ID'].tolist())
print(f'\n[1] Phage metadata — expected {len(expected_phage_ids)}, found {len(found_phage_ids)}')
if found_phage_ids == expected_phage_ids:
    print('    ✅ All test phages present in database')
else:
    missing = expected_phage_ids - found_phage_ids
    extra = found_phage_ids - expected_phage_ids
    if missing:
        print(f'    ❌ Missing phages: {missing}')
    if extra:
        print(f'    ⚠️  Unexpected phages: {extra}')
    if not found_phage_ids:
        print('    ⚠️  No test_private phages found — has the pipeline been run?')

# 2. Retrieve sequences for test_private pairs
print('\n[2] Sequence retrieval for test_private pairs...')
test_pairs = retriever.get_phage_host_pairs(where_clause="p.Source_DB = 'test_private'")
print(f'    Retrieved {len(test_pairs)} pairs')

if len(test_pairs) == 0:
    print('    ⚠️  No pairs returned — database may not include test_private data.')
    print('       Run the pipeline with private_data/test_private/ present.')
else:
    n_with_phage = (test_pairs['Phage_Sequence'].notna() & (test_pairs['Phage_Sequence'] != '')).sum()
    n_with_host = (test_pairs['Host_Sequence'].notna() & (test_pairs['Host_Sequence'] != '')).sum()
    print(f'    Phage sequences retrieved: {n_with_phage} / {len(test_pairs)}')
    print(f'    Host sequences retrieved:  {n_with_host} / {len(test_pairs)}')

    src_values = sorted(test_pairs['Phage_Source_Type'].dropna().astype(str).str.lower().unique())
    if src_values == ['private']:
        print('    ✅ Phage_Source_Type is consistently private')
    else:
        print(f'    ⚠️  Unexpected Phage_Source_Type values: {src_values}')

    display(test_pairs[['Phage_ID', 'Host_ID', 'Phage_Source_Type', 'Phage_Sequence', 'Host_Sequence']].head())

# 3. Spot-check TEST_PHAGE_ALPHA
print('\n[3] Spot-check: fetch TEST_PHAGE_ALPHA sequence directly')
alpha_pairs = retriever.get_phage_host_pairs(where_clause="p.Phage_ID = 'TEST_PHAGE_ALPHA'")
if len(alpha_pairs) > 0:
    seq = alpha_pairs.iloc[0]['Phage_Sequence']
    if seq:
        print(f'    ✅ TEST_PHAGE_ALPHA sequence length: {len(seq)} bp')
        print(f'       First 60 bp: {seq[:60]}')
    else:
        print('    ⚠️  Sequence value is None or empty')
else:
    print('    ⚠️  TEST_PHAGE_ALPHA not found in sequence pairs')


## Summary

This notebook demonstrated all sequence retrieval methods in PBI:

- **`get_phage_metadata()`** - SQL-filtered phage metadata
- **`get_host_metadata()`** - SQL-filtered host metadata
- **`get_phage_host_metadata()`** - Metadata-only pair queries (fast)
- **`get_phage_host_pairs()`** - Pairs with DNA sequences (slower, loads FASTA)
- **`get_phage_host_pairs_iterator()`** - Batch iteration for large datasets
- **`get_host_genome()`** - Full host genome (multi-contig safe); modes: `"concat"`, `"first"`, `"list"`, `"dict"`
- **`get_host_genome_stats()`** - Contig statistics without loading sequences
- **`get_phage_genome()`** - Full phage genome (same mode API)
- **`assemble_genome()` / `get_genome_stats()`** - Low-level FASTA assembly utilities (`pbi.fasta_utils`)
- **`create_streaming_dataset()`** / **`create_indexed_dataset()`** - ML-ready datasets

### Key Architecture Points
- DuckDB provides fast SQL queries on metadata
- pyfaidx provides O(1) sequence access via `.fai` index files
- Multi-contig host assemblies are handled transparently by `get_host_genome()` and `host_contig_mode="concat"`
- Missing sequences are skipped gracefully with warnings
- All methods return pandas DataFrames (except iterator and datasets)

See `03_ml_streaming.ipynb` for machine learning applications.

## Analysis output layout (durable exports)
Generated artifacts are written to `NOTEBOOK_RESULTS_DIR`, rooted at `/results` in Docker (mounted from `./analysis_results`):
- `tables/` for exported DataFrames (`.parquet`)
- `figures/` for saved plots (`.png`)
This keeps source notebooks in `/workspace` clean while preserving reproducible outputs in `/results`.
Note: all currently open matplotlib figures are exported to `figures/` when running the export cell.


In [ ]:
# Export meaningful notebook artifacts to durable results storage
import pandas as pd

candidate_tables = {
    'phage_metadata': globals().get('phage_metadata'),
    'host_metadata': globals().get('host_metadata'),
    'pairs_meta': globals().get('pairs_meta'),
    'pairs': globals().get('pairs'),
    'complete_phages': globals().get('complete_phages'),
    'large_phages': globals().get('large_phages'),
    'sipho_pairs': globals().get('sipho_pairs'),
    'private_phages': globals().get('private_phages'),
    'test_pairs': globals().get('test_pairs'),
}

exported_tables = []
for name, value in candidate_tables.items():
    if isinstance(value, pd.DataFrame) and not value.empty:
        output_path = TABLES_DIR / f"{name}.parquet"
        value.to_parquet(output_path, index=False)
        exported_tables.append(output_path)

exported_figures = []
for fig_num in plt.get_fignums():
    fig_path = FIGURES_DIR / f"figure_{fig_num}.png"
    plt.figure(fig_num).savefig(fig_path, dpi=300, bbox_inches='tight')
    exported_figures.append(fig_path)

print(f"Exported {len(exported_tables)} table(s) to: {TABLES_DIR}")
for p in exported_tables:
    print(f" - {p.name}")

print(f"Exported {len(exported_figures)} figure(s) to: {FIGURES_DIR}")
for p in exported_figures:
    print(f" - {p.name}")



In [ ]:
# Cleanup
#retriever.close()
#print("Database connection closed")